In [0]:
%sql
select * from samples.nyctaxi.trips

In [0]:
display(spark.read.table("samples.nyctaxi.trips"))

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.mydatabrick

In [0]:
catalog = "workspace"
schema = "default"
volume = "mydatabrick"
download_url = "https://health.data.ny.gov/api/views/jxy9-yhdk/rows.csv"
file_name = "baby_names.csv"
table_name = "baby_names"
path_volume = "/Volumes/" + catalog + "/" + schema + "/" + volume
path_table = catalog + "." + schema
print(path_table) # Show the complete path
print(path_volume) # Show the complete path

In [0]:
dbutils.fs.ls("/Volumes/workspace/default/mydatabrick/")

In [0]:
df = spark.read.csv(f"{path_volume}/{file_name}",
  header=True,
  inferSchema=True,
  sep=",")

In [0]:
import pandas as pd

# Check if file exists in volume
file_path = f"{path_volume}/{file_name}"
file_exists = False

try:
    dbutils.fs.ls(file_path)
    file_exists = True
    print(f"File already exists at {file_path}")
except:
    print(f"File not found. Downloading {file_name}...")

# Download if needed
if not file_exists:
    # Download using pandas
    pdf = pd.read_csv(download_url)
    # Convert to Spark DataFrame and write to volume
    df_temp = spark.createDataFrame(pdf)
    df_temp.coalesce(1).write.mode("overwrite").option("header", "true").csv(file_path)
    print(f"Downloaded to {file_path}")

# Read and display
df = spark.read.csv(file_path, header=True, inferSchema=True, sep=",")
display(df)

Databricks visualization. Run in Databricks to view.

In [0]:
df = df.withColumnRenamed("First Name", "First_Name")
df.printSchema

In [0]:
df.write.mode("overwrite").saveAsTable(f"{path_table}" + "." + f"{table_name}")